In [ ]:
# ==========================================
# 1. IMPORTS & HELPER FUNCTIONS
# ==========================================
import os
import re

def add_or_append_marc_field(file_path: str, tag: str, field_xml: str):
    """Appends a new MARC datafield into an existing XML record.
    
    If the specified tag already exists, the new field is inserted right below it.
    Otherwise, it is placed directly before the closing </record> tag.
    """
    if not os.path.exists(file_path):
        print(f"[-] File not found: {file_path}")
        return False

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        target_pattern = f'<datafield tag="{tag}"'
        
        if target_pattern in content:
            print(f"-> Existing {tag} field found. Appending below first instance...")
            pattern = rf'(<datafield tag="{tag}".*?</datafield>\n)'
            content = re.sub(pattern, r'\1' + field_xml, content, count=1, flags=re.DOTALL)
        else:
            print(f"-> No {tag} field found. Placing field before closing </record> tag...")
            content = content.replace("</record>", field_xml + "</record>")

        with open(file_path, "w", encoding="utf-8", newline="\n") as f:
            f.write(content)

        print(f"[+] {os.path.basename(file_path)} updated successfully.")
        return True

    except Exception as e:
        print(f"[-] Error processing {file_path}: {e}")
        return False


# ==========================================
# 2. CONFIGURATION & BATCH EDIT EXECUTION
# ==========================================
# Folder and target files
TEMPLATE_DIR = "gnd_edit_templates"
TARGET_FILES = [
    "ppn_5018471824_clean.xml",
    "ppn_5018471832_clean.xml",
    "ppn_5018471840_clean.xml"
]

# Field definition to insert
TARGET_TAG = "678"
NEW_FIELD_XML = (
    '  <datafield tag="678" ind1=" " ind2=" ">\n'
    '    <subfield code="b">Biographical or historical note text</subfield>\n'
    '  </datafield>\n'
)

print("Starting batch update of XML GND templates...\n")

for filename in TARGET_FILES:
    full_path = os.path.join(TEMPLATE_DIR, filename)
    add_or_append_marc_field(full_path, TARGET_TAG, NEW_FIELD_XML)

print("\nBatch processing complete. Verify updated files in your editor.")
